# Web Server Solutions

In [ ]:
import flask

import werkzeug

import psycopg2

import json


## You try it - add a route /stores that returns a web page with an HTML table with store id, city, total sales; solutions are in web_server_solutions and templates/stores_solutions.html

## You try it - add a route /stores/bootstrap that formats the results using Bootstrap; solutions are in web_server_solutions and templates/stores_bootstrap_solutions.html

## Try both on your laptop, desktop, phone, tablet, etc.


In [ ]:
connection = psycopg2.connect(
    user = "postgres",
    password = "ucb",
    host = "postgres",
    port = "5432",
    database = "postgres"
)

In [ ]:
cursor = connection.cursor()

In [ ]:
def my_query_products():
    "query the products from Postgres and return a Python list of products"
    
    connection.rollback()

    query = """
    
    select p.product_id, p.description, sum(quantity), sum(quantity * 12)
    from products p
         join line_items l
             on p.product_id = l.product_id
    group by p.product_id, p.description
    order by p.product_id
    
    """
    
    cursor.execute(query)
    
    rows = cursor.fetchall()

    connection.rollback()
    
    products_list = []
    
    for row in rows:
        
        products_list.append([row[0], row[1], f'{row[2]:,}', f'{row[3]:,}'])
        
    return(products_list)

In [ ]:
def my_query_stores():
    "query the stores from Postgres and return a Python list of stores"
    
    connection.rollback()

    query = """
    
    select s.store_id, s.city, sum(sa.total_amount)
    from stores as s
         join sales as sa
             on s.store_id = sa.store_id
    group by s.store_id, s.city
    order by s.store_id
    
    """
    
    cursor.execute(query)
    
    rows = cursor.fetchall()

    connection.rollback()
    
    stores_list = []
    
    for row in rows:
        
        stores_list.append([row[0], row[1], f'{row[2]:,}'])
        
    return(stores_list)

In [ ]:
app = flask.Flask(__name__,
                  static_url_path="")

@app.route("/")
def landing_page():
    return flask.send_from_directory("static", "index.html")

@app.route("/products")
def products():
    
    products_list = my_query_products()
    
    return(flask.render_template("products.html", products_list=products_list))

@app.route("/products/bootstrap")
def products_bootstrap():
    
    products_list = my_query_products()
    
    return(flask.render_template("products_bootstrap.html", products_list=products_list))

@app.route("/stores")
def stores():
    
    stores_list = my_query_stores()
    
    return(flask.render_template("stores_solutions.html", stores_list=stores_list))

@app.route("/stores/bootstrap")
def stores_bootstrap():
    
    stores_list = my_query_stores()
    
    return(flask.render_template("stores_bootstrap_solutions.html", stores_list=stores_list))


### We are running several web servers for this lab.  For reference, this is web_server_solutions.ipynb A

In [ ]:
werkzeug.serving.run_simple(hostname="0.0.0.0", 
                            port=443, 
                            application=app,
                            ssl_context=("w205_cert.pem","w205.key"),
                            use_debugger=True)

## You try it - add an API call for route /api/stores which returns the stores query results as JSON; solution is in web_server_solutions and web_client_solutions


In [ ]:
app = flask.Flask(__name__,
                  static_url_path="")

@app.route("/")
def landing_page():
    return flask.send_from_directory("static", "index.html")

@app.route("/products")
def products():
    
    products_list = my_query_products()
    
    return(flask.render_template("products.html", products_list=products_list))

@app.route("/products/bootstrap")
def products_bootstrap():
    
    products_list = my_query_products()
    
    return(flask.render_template("products_bootstrap.html", products_list=products_list))

@app.route("/stores")
def stores():
    
    stores_list = my_query_stores()
    
    return(flask.render_template("stores_solutions.html", stores_list=stores_list))

@app.route("/stores/bootstrap")
def stores_bootstrap():
    
    stores_list = my_query_stores()
    
    return(flask.render_template("stores_bootstrap_solutions.html", stores_list=stores_list))

@app.route("/api/products", methods=["GET"])
def api_products():
    
    products_list = my_query_products()
    
    products_json_list = []
    
    for product in products_list:
        
        p = {}
        p["product_id"] = str(product[0])
        p["product_name"] = product[1]
        p["quantity"] = str(product[2])
        p["total_sales"] = str(product[3])
        
        products_json_list.append(p)
        
    return(json.dumps(products_json_list))

@app.route("/api/stores", methods=["GET"])
def api_stores():
    
    stores_list = my_query_stores()
    
    stores_json_list = []
    
    for store in stores_list:
        
        s = {}
        s["store_id"] = str(store[0])
        s["city"] = store[1]
        s["total_sales"] = str(store[2])
        
        stores_json_list.append(s)
        
    return(json.dumps(stores_json_list))



### We are running several web servers for this lab.  For reference, this is web_server_solutions.ipynb B

In [ ]:
werkzeug.serving.run_simple(hostname="0.0.0.0", 
                            port=443, 
                            application=app,
                            ssl_context=("w205_cert.pem","w205.key"),
                            use_debugger=True)

## You try it - modify an API call for route /api/stores to have the option of passing a store parameter; solution is in web_server_solutions and web_client_solutions


In [ ]:
app = flask.Flask(__name__,
                  static_url_path="")

@app.route("/")
def landing_page():
    return flask.send_from_directory("static", "index.html")

@app.route("/products")
def products():
    
    products_list = my_query_products()
    
    return(flask.render_template("products.html", products_list=products_list))

@app.route("/products/bootstrap")
def products_bootstrap():
    
    products_list = my_query_products()
    
    return(flask.render_template("products_bootstrap.html", products_list=products_list))

@app.route("/stores")
def stores():
    
    stores_list = my_query_stores()
    
    return(flask.render_template("stores_solutions.html", stores_list=stores_list))

@app.route("/stores/bootstrap")
def stores_bootstrap():
    
    stores_list = my_query_stores()
    
    return(flask.render_template("stores_bootstrap_solutions.html", stores_list=stores_list))

@app.route("/api/products", methods=["GET"])
def api_products():
    
    products_list = my_query_products()
    
    products_json_list = []
    
    product_parameter = flask.request.args.get("product")
    
    for product in products_list:
        
        if product_parameter == None or (product_parameter != None and product_parameter == str(product[0])):
            
            p = {}
            p["product_id"] = str(product[0])
            p["product_name"] = product[1]
            p["quantity"] = str(product[2])
            p["total_sales"] = str(product[3])

            products_json_list.append(p)
        
    return(json.dumps(products_json_list))
    

@app.route("/api/stores", methods=["GET"])
def api_stores():
    
    stores_list = my_query_stores()
    
    stores_json_list = []
    
    store_parameter = flask.request.args.get("store")

    for store in stores_list:
        
        if store_parameter == None or (store_parameter != None and store_parameter == str(store[0])):
            
            s = {}
            s["store_id"] = str(store[0])
            s["city"] = store[1]
            s["total_sales"] = str(store[2])

            stores_json_list.append(s)

    return(json.dumps(stores_json_list))



### We are running several web servers for this lab.  For reference, this is web_server_solutions.ipynb C

In [ ]:
werkzeug.serving.run_simple(hostname="0.0.0.0", 
                            port=443, 
                            application=app,
                            ssl_context=("w205_cert.pem","w205.key"),
                            use_debugger=True)

## You try it - modify an API call for route /api/stores to have the option of passing a store parameter using JSON in a POST method; solution is in web_server_solutions and web_client_solutions


In [ ]:
app = flask.Flask(__name__,
                  static_url_path="")

@app.route("/")
def landing_page():
    return flask.send_from_directory("static", "index.html")

@app.route("/products")
def products():
    
    products_list = my_query_products()
    
    return(flask.render_template("products.html", products_list=products_list))

@app.route("/products/bootstrap")
def products_bootstrap():
    
    products_list = my_query_products()
    
    return(flask.render_template("products_bootstrap.html", products_list=products_list))

@app.route("/api/products", methods=["GET","POST"])
def api_products():
    
    products_list = my_query_products()
    
    products_json_list = []
    
    if flask.request.method == "GET":
        
        product_parameter = flask.request.args.get("product")
        
    elif flask.request.method == "POST":
        
        product_parameter = flask.request.form['product']
    
    for product in products_list:
        
        if product_parameter == None or (product_parameter != None and product_parameter == str(product[0])):
            
            p = {}
            p["product_id"] = str(product[0])
            p["product_name"] = product[1]
            p["quantity"] = str(product[2])
            p["total_sales"] = str(product[3])

            products_json_list.append(p)
        
    return(json.dumps(products_json_list))
    
@app.route("/api/stores", methods=["GET", "POST"])

def api_stores():
    
    stores_list = my_query_stores()
    
    stores_json_list = []
    
    if flask.request.method == "GET":
        
        store_parameter = flask.request.args.get("store")
        
    elif flask.request.method == "POST":
        
        store_parameter = flask.request.form['store']

    for store in stores_list:
        
        if store_parameter == None or (store_parameter != None and store_parameter == str(store[0])):
            
            s = {}
            s["store_id"] = str(store[0])
            s["city"] = store[1]
            s["total_sales"] = str(store[2])

            stores_json_list.append(s)

    return(json.dumps(stores_json_list))


### We are running several web servers for this lab.  For reference, this is web_server_solutions.ipynb D

In [ ]:
werkzeug.serving.run_simple(hostname="0.0.0.0", 
                            port=443, 
                            application=app,
                            ssl_context=("w205_cert.pem","w205.key"),
                            use_debugger=True)